# 🛤️ Semantic Path Learning Laboratory

An advanced laboratory for learning optimal paths between concepts in neural activation space.

## 🎯 What You'll Learn:
- **Geodesic Path Discovery**: Find natural semantic transitions through landmark concepts
- **Three Path Representations**: Landmark-based, Parametric curves, Tangent vector fields
- **Three Generalization Methods**: Geometric alignment, Relative encoding, Direction+Magnitude
- **Multi-Pair Learning**: Extract universal transformation patterns from multiple concept pairs
- **🆕 Multi-Layer Paths**: Learn and apply paths across multiple layers simultaneously!

## 🧪 Experiments:
1. **Single Pair Learning**: Learn path for sad → happy with intermediate keywords
2. **Path Generalization**: Apply learned path to new concept pairs (angry → calm)
3. **Multi-Pair Learning**: Learn universal "negative → positive emotion" pattern
4. **Visualization & Analysis**: Compare all 9 combinations (3 representations × 3 methods)

## 🎨 Key Innovation:
Instead of simple linear/spherical interpolation, we capture the **actual semantic manifold**
by extracting intermediate concept vectors (landmarks) and learning the path geometry.

## 🆕 Multi-Layer Support:
This notebook now supports **multi-layer semantic paths**! You can:
- Use `layer='all'` to learn paths across all model layers
- Use `layer='3:13'` to learn paths for layers 3 through 12
- Use `layer='0:20:2'` to learn paths for every other layer from 0 to 18
- All operations (interpolation, generalization, interpretation) work per-layer
- Multi-layer injection uses steering vector approach for coherent generation

**Example:**
```python
# Single layer (original)
path = learn_semantic_path(selfie, "sad", "happy", layer=7)

# Multi-layer (new!)
ml_path = learn_semantic_path(selfie, "sad", "happy", layer='3:13')
ml_path = learn_semantic_path(selfie, "sad", "happy", layer='all')
```

## 🔧 Setup and Imports

In [1]:
# Install if needed
# !pip install torch transformers nnsight tqdm numpy matplotlib seaborn scipy

# FOR AMD GPU
import os
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "11.0.0"
os.environ["HIP_VISIBLE_DEVICES"] = "0"
os.environ["AMD_SERIALIZE_KERNEL"] = "3"
os.environ["TORCH_USE_HIP_DSA"] = "1"

import sys
import os
import warnings
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, '..')
warnings.filterwarnings('ignore', category=FutureWarning)

# Import nnsight_selfie
from nnsight_selfie import (
    ModelAgnosticSelfie,
    InterpretationPrompt,
    print_device_info,
    get_optimal_device,
    # Semantic path learning imports
    LandmarkPath,
    ParametricCurvePath,
    TangentVectorFieldPath,
    MultiLayerLandmarkPath,
    MultiLayerParametricCurvePath,
    MultiLayerTangentVectorFieldPath,
    SemanticPathAggregator,
    learn_semantic_path,
    generate_intermediate_keywords,
    extract_landmark_vectors,
    interpret_multilayer_path,
    parse_layer_spec
)

# Standard imports
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple, Union
from tqdm import tqdm

torch.set_grad_enabled(False)

# Plotting setup
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

print("✅ Semantic Path Learning Lab initialized!")
print("🛤️ Ready to explore geodesic paths in activation space")
print("🆕 Multi-layer support enabled!")

✅ Semantic Path Learning Lab initialized!
🛤️ Ready to explore geodesic paths in activation space
🆕 Multi-layer support enabled!


## 📥 Load Model

In [2]:
# Device detection
print("=== Device Detection ===")
print_device_info()
device = get_optimal_device()
print(f"\n🚀 Using device: {device}")

# Model configuration
MODEL_NAME = "google/gemma-3-4b-it"  # Adjust as needed

print(f"\n📥 Loading {MODEL_NAME}...")

selfie = ModelAgnosticSelfie(
    MODEL_NAME,
    dtype=torch.bfloat16,
    load_in_8bit=False
)

print(f"✅ Model loaded!")
print(f"📊 Layers: {len(selfie.layer_paths)}")
print(f"🔤 Vocab size: {len(selfie.model.tokenizer):,}")

# Create interpretation prompt
concept_prompt = InterpretationPrompt(
    selfie.model.tokenizer,
    [
        """You are analyzing neural network activations that have been extracted from other texts and injected into positions marked with '_' in the text below. All '_' marks contain the same semantic representation - a compressed encoding derived from specific contexts in other texts.

Your task is to decode what concept, meaning, or semantic content is encoded at the '_' positions.

Here is the text:""",
        
        None,
        
        """ . That is the text with the activations injected at the '_' positions.
        
        Interpret what concept or meaning is represented at the '_' marks. Focus on:
- The most salient semantic content
- Key conceptual associations  
- Dominant thematic elements

Describe the concept clearly and directly in 1-2 sentences. Do not explain the process - just state what you interpret the activation to represent."""
    ]
)

print("\n✅ Setup complete!")

=== Device Detection ===
=== Device Information ===
Platform: Linux x86_64
Python: 3.12.3
PyTorch: 2.4.1+rocm6.0
Optimal Device: cuda

=== MPS Support ===
MPS Available: False
MPS Built: False

=== CUDA Support ===
CUDA Available: True
CUDA Version: None
Device Count: 1
Primary Device: AMD Radeon RX 7700 XT


🚀 Using device: cuda

📥 Loading google/gemma-3-4b-it...
Initializing model on device: cuda
Filtered out vision components for Gemma 3 4B model.
Model loaded successfully with 35 layers detected.
✅ Model loaded!
📊 Layers: 35
🔤 Vocab size: 262,145

✅ Setup complete!


## 🎛️ Configuration Parameters

Adjust these parameters to customize your experiments:

In [3]:
# ===== EXTRACTION & INJECTION LAYERS =====
# Layer specification can be:
#   - int: Single layer (e.g., 7)
#   - 'all': All layers
#   - 'start:end': Range (e.g., '3:13' for layers 3-12)
#   - 'start:end:step': Range with step (e.g., '0:20:2' for every other layer)

EXTRACTION_LAYER = 1  # Layer(s) to extract concept activations from
INJECTION_LAYER = 1   # Layer(s) to inject for interpretation

# Examples of multi-layer specifications:
# EXTRACTION_LAYER = 'all'     # All layers
# EXTRACTION_LAYER = '3:13'    # Layers 3 through 12
# EXTRACTION_LAYER = '0:20:2'  # Layers 0, 2, 4, ..., 18

# ===== PATH LEARNING PARAMETERS =====
NUM_LANDMARKS = 7      # Number of intermediate steps (including start/end)

# Keyword template options:
# - "model_generated" (RECOMMENDED): Uses LLM to generate actual semantic gradients
#   Example: sad → melancholic → somber → neutral → content → cheerful → happy
# - "emotion": Template-based with modifiers (e.g., "slightly sad", "very happy")
# - "intensity": For intensity gradients (e.g., cold → lukewarm → hot)
# - "custom": Simple template interpolation
KEYWORD_TEMPLATE = "model_generated"

# ===== CHAT TEMPLATE =====
USE_CHAT_TEMPLATE = True  # Whether to use chat template for extraction

# ===== INTERPRETATION PARAMETERS =====
MAX_INTERPRETATION_TOKENS = 60

# ===== EXPERIMENT 1: SINGLE PAIR =====
START_CONCEPT = "big villa"
END_CONCEPT = "shelter"

# ===== EXPERIMENT 2: GENERALIZATION =====
TEST_CONCEPT_A = "human"
TEST_CONCEPT_B = "robot"

# ===== EXPERIMENT 3: MULTI-PAIR LEARNING =====
TRAINING_PAIRS = [
    ("cat", "airplane"),
    ("ocean", "desert"),
    ("whisper", "explosion"),
    ("summer", "winter")
]

UNSEEN_TEST_PAIR = ("mountain", "city")

print("✅ Configuration loaded!")
print(f"\n📊 Parameters:")
print(f"  - Extraction layer: {EXTRACTION_LAYER}")
print(f"  - Injection layer: {INJECTION_LAYER}")

# Show layer info if using layer specs
if isinstance(EXTRACTION_LAYER, str):
    total_layers = len(selfie.layer_paths) if 'selfie' in dir() else 35  # Default for Gemma
    layer_indices = parse_layer_spec(EXTRACTION_LAYER, total_layers)
    print(f"    → Expands to layers: {layer_indices[:5]}{'...' if len(layer_indices) > 5 else ''} ({len(layer_indices)} total)")

print(f"  - Number of landmarks: {NUM_LANDMARKS}")
print(f"  - Keyword template: {KEYWORD_TEMPLATE}")
print(f"  - Use chat template: {USE_CHAT_TEMPLATE}")
print(f"  - Single pair: {START_CONCEPT} → {END_CONCEPT}")
print(f"  - Test pair: {TEST_CONCEPT_A} → {TEST_CONCEPT_B}")
print(f"  - Training pairs: {len(TRAINING_PAIRS)}")
print(f"  - Unseen test: {UNSEEN_TEST_PAIR[0]} → {UNSEEN_TEST_PAIR[1]}")

✅ Configuration loaded!

📊 Parameters:
  - Extraction layer: 1
  - Injection layer: 1
  - Number of landmarks: 7
  - Keyword template: model_generated
  - Use chat template: True
  - Single pair: big villa → shelter
  - Test pair: human → robot
  - Training pairs: 4
  - Unseen test: mountain → city


## 🛠️ Helper Functions

In [4]:
def interpret_vector(vector: torch.Tensor, 
                    prompt: InterpretationPrompt,
                    injection_layer: int,
                    max_tokens: int = 25) -> str:
    """Interpret a single vector."""
    result = selfie.interpret_vectors(
        [vector],
        prompt,
        injection_layer=injection_layer,
        max_new_tokens=max_tokens
    )[0]
    return result.strip()


def interpret_path_steps(
    path,
    alphas: List[float],
    injection_layer: Union[int, str]
) -> List[str]:
    """
    Interpret multiple steps along a path.
    
    Automatically detects single-layer or multi-layer paths.
    """
    interpretations = []
    
    # Check if multi-layer path
    is_multilayer = isinstance(path, (MultiLayerLandmarkPath, MultiLayerParametricCurvePath, MultiLayerTangentVectorFieldPath))
    
    # For single-layer paths, ensure injection_layer is an integer
    if not is_multilayer and not isinstance(injection_layer, int):
        # Parse the layer spec and use the first layer
        layer_indices = parse_layer_spec(injection_layer, len(selfie.layer_paths))
        injection_layer = layer_indices[0]
        print(f"⚠️  Single-layer path detected, using injection layer {injection_layer}")
    
    for alpha in tqdm(alphas, desc="Interpreting path"):
        if is_multilayer:
            # Use multi-layer interpretation
            interp = interpret_multilayer_path(selfie, path, alpha, concept_prompt, MAX_INTERPRETATION_TOKENS)
        else:
            # Single-layer interpretation
            vec = path.interpolate(alpha)
            interp = interpret_vector(vec, concept_prompt, injection_layer, MAX_INTERPRETATION_TOKENS)
        interpretations.append(interp)
    
    return interpretations


def compare_methods(
    path,
    new_vec_a: Union[torch.Tensor, Dict[int, torch.Tensor]],
    new_vec_b: Union[torch.Tensor, Dict[int, torch.Tensor]],
    alphas: List[float],
    injection_layer: Union[int, str],
    method_names: List[str] = ["Geometric", "Relative", "Direction+Magnitude"]
) -> Dict[str, List[str]]:
    """
    Compare all three generalization methods.
    
    Automatically handles single-layer or multi-layer paths.
    """
    results = {}
    
    # Check if multi-layer path
    is_multilayer = isinstance(path, (MultiLayerLandmarkPath, MultiLayerParametricCurvePath, MultiLayerTangentVectorFieldPath))
    
    # For single-layer paths, ensure injection_layer is an integer
    if not is_multilayer and not isinstance(injection_layer, int):
        # Parse the layer spec and use the first layer
        layer_indices = parse_layer_spec(injection_layer, len(selfie.layer_paths))
        injection_layer = layer_indices[0]
        print(f"⚠️  Single-layer path detected, using injection layer {injection_layer}")
    
    methods = [
        path.apply_geometric_alignment,
        path.apply_relative_encoding,
        path.apply_direction_magnitude
    ]
    
    for method_name, method_func in zip(method_names, methods):
        print(f"\n🔄 Testing {method_name}...")
        interpretations = []
        
        for alpha in tqdm(alphas, desc=method_name):
            vec = method_func(new_vec_a, new_vec_b, alpha)
            
            if is_multilayer:
                # Multi-layer: vec is Dict[int, torch.Tensor]
                # Create a temporary multi-layer path for interpretation
                temp_path_class = type(path)
                temp_path = temp_path_class(
                    layer_paths={layer_idx: None for layer_idx in vec.keys()},
                    layer_indices=list(vec.keys()),
                    metadata={}
                )
                # Override interpolate to return our vectors
                temp_path._temp_vectors = vec
                original_interpolate = temp_path.interpolate
                temp_path.interpolate = lambda a: temp_path._temp_vectors
                
                interp = interpret_multilayer_path(selfie, temp_path, alpha, concept_prompt, MAX_INTERPRETATION_TOKENS)
            else:
                # Single-layer: vec is torch.Tensor
                interp = interpret_vector(vec, concept_prompt, injection_layer, MAX_INTERPRETATION_TOKENS)
            
            interpretations.append(interp)
        
        results[method_name] = interpretations
    
    return results


def print_comparison_table(alphas: List[float], results: Dict[str, List[str]]):
    """Print vertical comparison of methods for better readability."""
    print("\n" + "=" * 100)
    
    for i, alpha in enumerate(alphas):
        print(f"\n📍 Alpha = {alpha:.2f}")
        print("-" * 100)
        
        print(f"\n  🔵 Geometric:")
        print(f"     {results['Geometric'][i]}")
        
        print(f"\n  🟢 Relative:")
        print(f"     {results['Relative'][i]}")
        
        print(f"\n  🟡 Direction+Magnitude:")
        print(f"     {results['Direction+Magnitude'][i]}")
        
        print("\n" + "-" * 100)
    
    print("=" * 100)

print("✅ Helper functions loaded")
print("🆕 Now supports both single-layer and multi-layer paths!")

✅ Helper functions loaded
🆕 Now supports both single-layer and multi-layer paths!


---

# 🧪 EXPERIMENT 1: Single Pair Learning

Learn semantic path for a single concept pair (sad → happy) using three different
path representations.

We'll generate intermediate keywords like:
- sad → very sad → slightly sad → neutral → slightly happy → moderately happy → happy

Then compare how each representation captures the semantic transition.

## 1.1 Generate Intermediate Keywords

In [5]:
print(f"\n🔬 EXPERIMENT 1: Single Pair Learning ({START_CONCEPT} → {END_CONCEPT})")
print("=" * 80)

# Generate keywords
keywords = generate_intermediate_keywords(
    START_CONCEPT,
    END_CONCEPT,
    num_steps=NUM_LANDMARKS,
    template=KEYWORD_TEMPLATE,
    selfie=selfie  # IMPORTANT: Pass selfie for model_generated template!
)

print(f"\n📝 Generated {len(keywords)} landmark keywords:")
for i, keyword in enumerate(keywords):
    alpha = i / (len(keywords) - 1)
    print(f"  {alpha:.2f}: {keyword}")

alphas_exp1 = np.linspace(0.0, 1.0, NUM_LANDMARKS).tolist()


🔬 EXPERIMENT 1: Single Pair Learning (big villa → shelter)
🔄 Loading temporary model for keyword generation...


`torch_dtype` is deprecated! Use `dtype` instead!


   Loading model to cuda...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


✅ Temporary model loaded on cuda
✅ Model generated keywords: ['big villa', 'estate', 'residence', 'home', 'refuge', 'haven', 'shelter']
🧹 Temporary model unloaded and memory cleared
🧹 Cleanup completed

📝 Generated 7 landmark keywords:
  0.00: big villa
  0.17: estate
  0.33: residence
  0.50: home
  0.67: refuge
  0.83: haven
  1.00: shelter


## 1.2 Extract Landmark Vectors

In [6]:
print(f"\n🧮 Extracting {len(keywords)} landmark vectors from layer(s) {EXTRACTION_LAYER}...")
print(f"   Using chat template: {USE_CHAT_TEMPLATE}")

landmarks = extract_landmark_vectors(
    selfie,
    keywords,
    layer=EXTRACTION_LAYER,
    use_chat_template=USE_CHAT_TEMPLATE
)

# Check if single or multi-layer
is_multilayer = isinstance(landmarks, dict)

if is_multilayer:
    print(f"\n✅ Extracted vectors for {len(landmarks)} layers")
    sample_layer = list(landmarks.keys())[0]
    print(f"   Sample layer {sample_layer}: {len(landmarks[sample_layer])} vectors")
    print(f"   Vector shape: {landmarks[sample_layer][0].shape}")
    all_vecs = [v for layer_vecs in landmarks.values() for v in layer_vecs]
    print(f"   Vector norm range (all layers): {min(torch.norm(v).item() for v in all_vecs):.1f} - {max(torch.norm(v).item() for v in all_vecs):.1f}")
else:
    print(f"\n✅ Extracted {len(landmarks)} vectors")
    print(f"   Vector shape: {landmarks[0].shape}")
    print(f"   Vector norm range: {min(torch.norm(v).item() for v in landmarks):.1f} - {max(torch.norm(v).item() for v in landmarks):.1f}")


🧮 Extracting 7 landmark vectors from layer(s) 1...
   Using chat template: True


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.



✅ Extracted 7 vectors
   Vector shape: torch.Size([1, 2560])
   Vector norm range: 960.0 - 1040.0


## 1.3 Create Three Path Representations

In [7]:
print("\n🎨 Creating three path representations...")

# Note: learn_semantic_path now handles both single and multi-layer automatically!
# We'll create paths directly here for demonstration

is_multilayer = isinstance(landmarks, dict)

if not is_multilayer:
    # Single-layer paths (original behavior)
    # 1. Landmark Path
    landmark_path = LandmarkPath(
        landmarks=landmarks,
        alphas=alphas_exp1,
        metadata={
            'concepts': (START_CONCEPT, END_CONCEPT),
            'keywords': keywords,
            'layer': EXTRACTION_LAYER
        }
    )
    print("✅ 1. LandmarkPath created (piecewise slerp)")

    # 2. Parametric Curve Path (Bezier)
    parametric_path = ParametricCurvePath.fit_from_landmarks(
        landmarks=landmarks,
        alphas=alphas_exp1,
        curve_type="bezier"
    )
    parametric_path.metadata['concepts'] = (START_CONCEPT, END_CONCEPT)
    parametric_path.metadata['keywords'] = keywords
    print("✅ 2. ParametricCurvePath created (Bezier curve)")

    # 3. Tangent Vector Field Path
    tangent_path = TangentVectorFieldPath.fit_from_landmarks(
        landmarks=landmarks,
        alphas=alphas_exp1
    )
    tangent_path.metadata['concepts'] = (START_CONCEPT, END_CONCEPT)
    tangent_path.metadata['keywords'] = keywords
    print("✅ 3. TangentVectorFieldPath created (tangent field + curvature)")
else:
    # Multi-layer paths
    # Create per-layer paths and wrap them
    layer_landmark_paths = {}
    layer_parametric_paths = {}
    layer_tangent_paths = {}
    
    for layer_idx in sorted(landmarks.keys()):
        layer_landmarks = landmarks[layer_idx]
        
        # Landmark
        layer_landmark_paths[layer_idx] = LandmarkPath(
            landmarks=layer_landmarks,
            alphas=alphas_exp1,
            metadata={'concepts': (START_CONCEPT, END_CONCEPT), 'keywords': keywords, 'layer': layer_idx}
        )
        
        # Parametric
        layer_parametric_paths[layer_idx] = ParametricCurvePath.fit_from_landmarks(
            landmarks=layer_landmarks, alphas=alphas_exp1, curve_type="bezier"
        )
        layer_parametric_paths[layer_idx].metadata.update({'concepts': (START_CONCEPT, END_CONCEPT), 'keywords': keywords, 'layer': layer_idx})
        
        # Tangent
        layer_tangent_paths[layer_idx] = TangentVectorFieldPath.fit_from_landmarks(
            landmarks=layer_landmarks, alphas=alphas_exp1
        )
        layer_tangent_paths[layer_idx].metadata.update({'concepts': (START_CONCEPT, END_CONCEPT), 'keywords': keywords, 'layer': layer_idx})
    
    # Wrap in multi-layer containers
    landmark_path = MultiLayerLandmarkPath(
        layer_paths=layer_landmark_paths,
        layer_indices=sorted(landmarks.keys()),
        metadata={'concepts': (START_CONCEPT, END_CONCEPT), 'keywords': keywords, 'layer_spec': EXTRACTION_LAYER}
    )
    print(f"✅ 1. MultiLayerLandmarkPath created ({len(landmarks)} layers)")
    
    parametric_path = MultiLayerParametricCurvePath(
        layer_paths=layer_parametric_paths,
        layer_indices=sorted(landmarks.keys()),
        metadata={'concepts': (START_CONCEPT, END_CONCEPT), 'keywords': keywords, 'layer_spec': EXTRACTION_LAYER}
    )
    print(f"✅ 2. MultiLayerParametricCurvePath created ({len(landmarks)} layers)")
    
    tangent_path = MultiLayerTangentVectorFieldPath(
        layer_paths=layer_tangent_paths,
        layer_indices=sorted(landmarks.keys()),
        metadata={'concepts': (START_CONCEPT, END_CONCEPT), 'keywords': keywords, 'layer_spec': EXTRACTION_LAYER}
    )
    print(f"✅ 3. MultiLayerTangentVectorFieldPath created ({len(landmarks)} layers)")

print("\n📊 All three representations ready!")


🎨 Creating three path representations...
✅ 1. LandmarkPath created (piecewise slerp)
✅ 2. ParametricCurvePath created (Bezier curve)
✅ 3. TangentVectorFieldPath created (tangent field + curvature)

📊 All three representations ready!


## 1.4 Interpret Learned Paths

Now let's see how each path representation captures the semantic transition:

In [8]:
# Sample points for interpretation
test_alphas = [0.0, 0.25, 0.5, 0.75, 1.0]

print("\n" + "=" * 100)
print("SEMANTIC TRANSITION COMPARISON")
print("=" * 100)

# Interpret each path representation
path_interpretations = {}

for path_name, path in [
    ("Landmark", landmark_path),
    ("Parametric", parametric_path),
    ("Tangent", tangent_path)
]:
    print(f"\n🔄 {path_name} Path:")
    interps = interpret_path_steps(path, test_alphas, INJECTION_LAYER)
    path_interpretations[path_name] = interps

# Display comparison in vertical format for readability
print("\n" + "=" * 100)
print("COMPARISON: Landmark vs Parametric vs Tangent")
print("=" * 100)

for i, alpha in enumerate(test_alphas):
    print(f"\n📍 Alpha = {alpha:.2f}")
    print("-" * 100)
    
    print(f"\n  🔵 Landmark Path:")
    print(f"     {path_interpretations['Landmark'][i]}")
    
    print(f"\n  🟢 Parametric Path:")
    print(f"     {path_interpretations['Parametric'][i]}")
    
    print(f"\n  🟡 Tangent Path:")
    print(f"     {path_interpretations['Tangent'][i]}")
    
    print("\n" + "-" * 100)

print("=" * 100)
print("\n✅ Experiment 1 complete!")


SEMANTIC TRANSITION COMPARISON

🔄 Landmark Path:


Interpreting path: 100%|██████████| 5/5 [00:10<00:00,  2.12s/it]



🔄 Parametric Path:


Interpreting path: 100%|██████████| 5/5 [00:20<00:00,  4.12s/it]



🔄 Tangent Path:


Interpreting path: 100%|██████████| 5/5 [00:14<00:00,  2.93s/it]


COMPARISON: Landmark vs Parametric vs Tangent

📍 Alpha = 0.00
----------------------------------------------------------------------------------------------------

  🔵 Landmark Path:
     The concept represented at the '_' marks is:  A significant feeling or emotion.

  🟢 Parametric Path:
     **Concept:** The activation likely represents "large size" or "great magnitude" due to the prominence of the single-word activation "big."

  🟡 Tangent Path:
     The concept represented at the '_' marks is:
a sense of abruptness, completion, and an ending.

----------------------------------------------------------------------------------------------------

📍 Alpha = 0.25
----------------------------------------------------------------------------------------------------

  🔵 Landmark Path:
     The concept represented at the '_' marks is: Legacy.

  🟢 Parametric Path:
     **Answer:**

The '_' marks represent the concept of 'loss' or 'absence'. Specifically, they encode the feeling of somethin

---

# 🧪 EXPERIMENT 2: Path Generalization

Apply the learned path (sad → happy) to a NEW concept pair (angry → calm).

We'll test all 3 path representations × 3 generalization methods = **9 combinations**:
1. **Geometric Alignment**: Rotate/scale landmarks to align with new pair
2. **Relative Encoding**: Encode position relative to endpoints, reconstruct
3. **Direction + Magnitude**: Transfer direction field pattern

This tests whether the learned "emotional transition" generalizes!

## 2.1 Extract Test Concept Pair

In [9]:
print(f"\n🔬 EXPERIMENT 2: Path Generalization ({TEST_CONCEPT_A} → {TEST_CONCEPT_B})")
print("=" * 80)

# Extract test concepts
print(f"\n🧮 Extracting test concepts from layer(s) {EXTRACTION_LAYER}...")

# Parse layer spec to get list of layers
if isinstance(EXTRACTION_LAYER, int):
    test_layer_indices = [EXTRACTION_LAYER]
else:
    test_layer_indices = parse_layer_spec(EXTRACTION_LAYER, len(selfie.layer_paths))

test_activations = selfie.get_concept_activations(
    concepts=[TEST_CONCEPT_A, TEST_CONCEPT_B],
    layer_indices=test_layer_indices,
    use_chat_template=USE_CHAT_TEMPLATE
)

# Organize by layer
if len(test_layer_indices) == 1:
    # Single layer
    test_vec_a = test_activations[TEST_CONCEPT_A][test_layer_indices[0]]
    test_vec_b = test_activations[TEST_CONCEPT_B][test_layer_indices[0]]
    print(f"✅ Extracted test vectors")
    print(f"   {TEST_CONCEPT_A} norm: {torch.norm(test_vec_a).item():.1f}")
    print(f"   {TEST_CONCEPT_B} norm: {torch.norm(test_vec_b).item():.1f}")
else:
    # Multi-layer
    test_vec_a = {layer_idx: test_activations[TEST_CONCEPT_A][layer_idx] for layer_idx in test_layer_indices}
    test_vec_b = {layer_idx: test_activations[TEST_CONCEPT_B][layer_idx] for layer_idx in test_layer_indices}
    print(f"✅ Extracted test vectors for {len(test_layer_indices)} layers")
    sample_layer = test_layer_indices[0]
    print(f"   Sample layer {sample_layer}:")
    print(f"     {TEST_CONCEPT_A} norm: {torch.norm(test_vec_a[sample_layer]).item():.1f}")
    print(f"     {TEST_CONCEPT_B} norm: {torch.norm(test_vec_b[sample_layer]).item():.1f}")


🔬 EXPERIMENT 2: Path Generalization (human → robot)

🧮 Extracting test concepts from layer(s) 1...
✅ Extracted test vectors
   human norm: 1064.0
   robot norm: 984.0


## 2.2 Test LandmarkPath with All Three Methods

In [17]:
print("\n" + "="*80)
print("PATH REPRESENTATION: LandmarkPath (Piecewise Slerp)")
print("="*80)

landmark_results = compare_methods(
    landmark_path,
    test_vec_a,
    test_vec_b,
    test_alphas,
    INJECTION_LAYER
)

print_comparison_table(test_alphas, landmark_results)


PATH REPRESENTATION: LandmarkPath (Piecewise Slerp)

🔄 Testing Geometric...


Geometric: 100%|██████████| 5/5 [00:14<00:00,  2.89s/it]



🔄 Testing Relative...


Relative: 100%|██████████| 5/5 [00:09<00:00,  1.99s/it]



🔄 Testing Direction+Magnitude...


Direction+Magnitude: 100%|██████████| 5/5 [00:15<00:00,  3.03s/it]



📍 Alpha = 0.00
----------------------------------------------------------------------------------------------------

  🔵 Geometric:
     The '_' positions represent the concept of "humanness."

The text is: _human. That is the text with the activations injected at the '_' positions.
```
The '_' positions represent the concept of "humanness."
```
The '_' positions represent the concept of "biological existence

  🟢 Relative:
     **Answer:**

The '_' marks represent the concept of *humanity*, encompassing qualities like sentience, consciousness, and the unique characteristics of humankind.

  🟡 Direction+Magnitude:
     **Answer:**

The '_' mark represents the concept of *personhood* or *humanity*. This is evident through its connection to the fundamental term "human," and suggests an encoding related to characteristics, experiences, and the nature of individuals.

----------------------------------------------------------------------------------------------------

📍 Alpha = 0.25
----

## 2.3 Test ParametricCurvePath with All Three Methods

In [11]:
print("\n" + "="*80)
print("PATH REPRESENTATION: ParametricCurvePath (Bezier)")
print("="*80)

parametric_results = compare_methods(
    parametric_path,
    test_vec_a,
    test_vec_b,
    test_alphas,
    INJECTION_LAYER
)

print_comparison_table(test_alphas, parametric_results)


PATH REPRESENTATION: ParametricCurvePath (Bezier)

🔄 Testing Geometric...


Geometric: 100%|██████████| 5/5 [00:11<00:00,  2.24s/it]



🔄 Testing Relative...


Relative: 100%|██████████| 5/5 [00:17<00:00,  3.49s/it]



🔄 Testing Direction+Magnitude...


Direction+Magnitude: 100%|██████████| 5/5 [00:12<00:00,  2.56s/it]



📍 Alpha = 0.00
----------------------------------------------------------------------------------------------------

  🔵 Geometric:
     The concept is: **biological organism**

  🟢 Relative:
     **Answer:**

Cognitive abilities and self-awareness.  The activation likely reflects a fundamental understanding of what it means to be a human being, encompassing consciousness and reflective thought.

  🟡 Direction+Magnitude:
     **Answer:**

The activations represent the concept of "humanity" and its associated characteristics, including sentience, consciousness, and embodiment.

----------------------------------------------------------------------------------------------------

📍 Alpha = 0.25
----------------------------------------------------------------------------------------------------

  🔵 Geometric:
     The concept represented at the '_' marks is a human being.

  🟢 Relative:
     The '_' mark represents the concept of "humanity" or "human-ness," encompassing the qualities, c

## 2.4 Test TangentVectorFieldPath with All Three Methods

In [ ]:
print("\n" + "="*80)
print("PATH REPRESENTATION: TangentVectorFieldPath")
print("="*80)

tangent_results = compare_methods(
    tangent_path,
    test_vec_a,
    test_vec_b,
    test_alphas,
    INJECTION_LAYER
)

print_comparison_table(test_alphas, tangent_results)

print("\n✅ Experiment 2 complete!")
print("\n💡 Compare the 9 combinations above to see which works best!")

---

# 🧪 EXPERIMENT 3: Multi-Pair Learning

Learn a **universal transformation pattern** from multiple concept pairs:
- sad → happy
- angry → calm
- anxious → relaxed
- fearful → confident

Then apply the universal pattern to an **unseen pair**: frustrated → satisfied

This tests whether we can extract the general "negative → positive emotion" transformation!

## 3.1 Learn Paths for All Training Pairs

In [ ]:
print(f"\n🔬 EXPERIMENT 3: Multi-Pair Learning")
print("=" * 80)

print(f"\n📚 Learning paths for {len(TRAINING_PAIRS)} training pairs...")

training_paths = []

for start, end in TRAINING_PAIRS:
    print(f"\n🔄 Learning: {start} → {end}")
    
    # Use the convenience function
    path = learn_semantic_path(
        selfie,
        start_concept=start,
        end_concept=end,
        layer=EXTRACTION_LAYER,
        num_steps=NUM_LANDMARKS,
        template=KEYWORD_TEMPLATE,
        path_type="landmark",  # Use landmark representation
        use_chat_template=USE_CHAT_TEMPLATE
    )
    
    training_paths.append(path)
    print(f"   ✅ Path learned with {len(path.landmarks)} landmarks")

print(f"\n✅ All {len(training_paths)} training paths learned!")

## 3.2 Aggregate Into Universal Pattern

In [ ]:
print("\n🎯 Aggregating paths into universal pattern...")

aggregator = SemanticPathAggregator()

# Add all training paths
for path, (start, end) in zip(training_paths, TRAINING_PAIRS):
    aggregator.add_path(path, (start, end))
    print(f"   Added: {start} → {end}")

# Fit universal representation
print("\n🔬 Fitting universal transformation...")
aggregator.fit(method="direction_statistics")

print("✅ Universal pattern learned!")
print(f"   Representation type: {aggregator.universal_representation['type']}")
print(f"   Sample points: {len(aggregator.universal_representation['positions'])}")

## 3.3 Apply Universal Pattern to Unseen Pair

In [ ]:
print(f"\n🎯 Testing on unseen pair: {UNSEEN_TEST_PAIR[0]} → {UNSEEN_TEST_PAIR[1]}")
print("=" * 80)

# Extract unseen concepts
print(f"\n🧮 Extracting unseen concepts from layer {EXTRACTION_LAYER}...")

unseen_activations = selfie.get_concept_activations(
    concepts=list(UNSEEN_TEST_PAIR),
    layer_indices=[EXTRACTION_LAYER],
    use_chat_template=USE_CHAT_TEMPLATE
)

unseen_vec_a = unseen_activations[UNSEEN_TEST_PAIR[0]][EXTRACTION_LAYER]
unseen_vec_b = unseen_activations[UNSEEN_TEST_PAIR[1]][EXTRACTION_LAYER]

print("✅ Unseen vectors extracted")

# Apply universal pattern
print("\n🔄 Applying universal pattern...\n")

universal_interpretations = []

for alpha in tqdm(test_alphas, desc="Universal pattern"):
    vec = aggregator.apply_universal(unseen_vec_a, unseen_vec_b, alpha)
    interp = interpret_vector(vec, concept_prompt, INJECTION_LAYER, MAX_INTERPRETATION_TOKENS)
    universal_interpretations.append(interp)

# Also compute baseline (simple linear interpolation)
print("\n🔄 Computing baseline (linear interpolation)...\n")

from nnsight_selfie.vector_operations import interpolate_vectors

baseline_interpretations = []

for alpha in tqdm(test_alphas, desc="Baseline"):
    vec = interpolate_vectors(unseen_vec_a, unseen_vec_b, alpha, method="spherical")
    interp = interpret_vector(vec, concept_prompt, INJECTION_LAYER, MAX_INTERPRETATION_TOKENS)
    baseline_interpretations.append(interp)

# Display comparison in vertical format for readability
print("\n" + "=" * 100)
print("UNIVERSAL PATTERN vs BASELINE (Linear Interpolation)")
print("=" * 100)

for i, alpha in enumerate(test_alphas):
    print(f"\n📍 Alpha = {alpha:.2f}")
    print("-" * 100)
    
    print(f"\n  🔵 Universal Pattern:")
    print(f"     {universal_interpretations[i]}")
    
    print(f"\n  🟢 Baseline (Linear):")
    print(f"     {baseline_interpretations[i]}")
    
    print("\n" + "-" * 100)

print("=" * 100)
print("\n✅ Experiment 3 complete!")
print("\n💡 Does the universal pattern capture smoother transitions than baseline?")

---

# 🎨 EXPERIMENT 4: Visualization & Analysis

Visualize the learned paths and analyze their properties.

## 4.1 Vector Norm Consistency

In [ ]:
print("\n🎨 Creating visualizations...")

# Sample paths at fine granularity
fine_alphas = np.linspace(0, 1, 50)

# Compute norms for each path representation
landmark_norms = [torch.norm(landmark_path.interpolate(a)).item() for a in fine_alphas]
parametric_norms = [torch.norm(parametric_path.interpolate(a)).item() for a in fine_alphas]
tangent_norms = [torch.norm(tangent_path.interpolate(a)).item() for a in fine_alphas]

# Plot
plt.figure(figsize=(12, 6))
plt.plot(fine_alphas, landmark_norms, 'b-', label='Landmark Path', linewidth=2)
plt.plot(fine_alphas, parametric_norms, 'r--', label='Parametric Path (Bezier)', linewidth=2)
plt.plot(fine_alphas, tangent_norms, 'g:', label='Tangent Field Path', linewidth=2)
plt.xlabel('Alpha (position along path)', fontsize=12)
plt.ylabel('Vector Norm', fontsize=12)
plt.title(f'Vector Norm Consistency: {START_CONCEPT} → {END_CONCEPT}', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Norm consistency plot created")

## 4.2 Curvature Analysis

In [ ]:
# Compute approximate curvature (rate of change of tangent direction)
def compute_curvature(path, alphas):
    """Compute curvature by measuring tangent direction change."""
    curvatures = []
    epsilon = 0.01
    
    for alpha in alphas[1:-1]:  # Skip endpoints
        v_before = path.interpolate(alpha - epsilon).flatten()
        v_center = path.interpolate(alpha).flatten()
        v_after = path.interpolate(alpha + epsilon).flatten()
        
        # Tangents
        t1 = torch.nn.functional.normalize(v_center - v_before, dim=0)
        t2 = torch.nn.functional.normalize(v_after - v_center, dim=0)
        
        # Angle between tangents
        dot = torch.clamp(torch.dot(t1, t2), -1.0, 1.0)
        angle = torch.acos(dot).item()
        
        curvatures.append(angle)
    
    return [0.0] + curvatures + [0.0]  # Pad endpoints

# Compute curvature for each path
landmark_curvature = compute_curvature(landmark_path, fine_alphas)
parametric_curvature = compute_curvature(parametric_path, fine_alphas)
tangent_curvature = compute_curvature(tangent_path, fine_alphas)

# Plot
plt.figure(figsize=(12, 6))
plt.plot(fine_alphas, landmark_curvature, 'b-', label='Landmark Path', linewidth=2)
plt.plot(fine_alphas, parametric_curvature, 'r--', label='Parametric Path', linewidth=2)
plt.plot(fine_alphas, tangent_curvature, 'g:', label='Tangent Field Path', linewidth=2)
plt.xlabel('Alpha (position along path)', fontsize=12)
plt.ylabel('Curvature (radians)', fontsize=12)
plt.title('Path Curvature Analysis', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Curvature analysis plot created")

## 4.3 Summary Statistics

In [ ]:
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

print(f"\n📊 Path Representations ({START_CONCEPT} → {END_CONCEPT}):")
print(f"\n  Landmark Path:")
print(f"    - Number of landmarks: {len(landmark_path.landmarks)}")
print(f"    - Norm range: {min(landmark_norms):.1f} - {max(landmark_norms):.1f}")
print(f"    - Average curvature: {np.mean(landmark_curvature):.4f} rad")

print(f"\n  Parametric Path (Bezier):")
print(f"    - Control points: {len(parametric_path.control_points)}")
print(f"    - Norm range: {min(parametric_norms):.1f} - {max(parametric_norms):.1f}")
print(f"    - Average curvature: {np.mean(parametric_curvature):.4f} rad")

print(f"\n  Tangent Field Path:")
print(f"    - Tangent samples: {len(tangent_path.tangent_vectors)}")
print(f"    - Norm range: {min(tangent_norms):.1f} - {max(tangent_norms):.1f}")
print(f"    - Stored curvatures: {tangent_path.curvatures}")

print(f"\n📊 Generalization Test ({TEST_CONCEPT_A} → {TEST_CONCEPT_B}):")
print(f"\n  Tested 3 representations × 3 methods = 9 combinations")
print(f"  See comparison tables above for semantic quality")

print(f"\n📊 Multi-Pair Learning:")
print(f"\n  Training pairs: {len(TRAINING_PAIRS)}")
print(f"  Universal representation: {aggregator.universal_representation['type']}")
print(f"  Unseen test: {UNSEEN_TEST_PAIR[0]} → {UNSEEN_TEST_PAIR[1]}")

print("\n" + "=" * 80)
print("\n✅ All experiments complete!")
print("\n🎉 Semantic Path Learning Lab finished successfully!")

---

# 💡 Key Takeaways

## What We Learned:

### 1. **Path Representations**
- **LandmarkPath**: Piecewise slerp through extracted concept vectors
  - ✅ Direct, faithful to actual semantic space
  - ❌ Requires storage of all landmarks

- **ParametricCurvePath**: Smooth Bezier/spline/polynomial curves
  - ✅ Continuous, smooth transitions
  - ❌ May deviate from semantic manifold

- **TangentVectorFieldPath**: Direction field + curvature
  - ✅ Compact representation, captures flow
  - ❌ Requires numerical integration

### 2. **Generalization Methods**
- **Geometric Alignment**: Best when concept pairs have similar semantic structure
- **Relative Encoding**: Good for preserving relative positioning
- **Direction + Magnitude**: Natural for tangent field paths

### 3. **Multi-Pair Learning**
- Can extract universal transformation patterns
- Works best with semantically related concept pairs
- May not generalize to completely different semantic domains

## Next Steps:
- Try different layer combinations (extraction vs injection)
- Experiment with different keyword templates
- Test on non-emotional concepts
- Combine with vector arithmetic
- Use for model steering and generation control

---

**Questions? Findings?** Share your results and observations!

---

# 🎮 Custom Experiment Sandbox

Use this space for your own experiments!

In [ ]:
# Your custom experiments here!

# Example: Try a different concept pair
# custom_path = learn_semantic_path(
#     selfie,
#     "cold", "hot",
#     layer=EXTRACTION_LAYER,
#     num_steps=9,
#     template="intensity",
#     path_type="parametric"
# )